***
<a id='beginning'></a>

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 建议先阅读： [8.1 作为最小二乘问题的校准](8_1_calibration_least_squares_problem.ipynb)

***


## 校准习题

本习题页把第 8 章的核心概念集中到一组合成观测中：五面天线、十条基线、三个点源、时间变化复增益和少量热噪声。练习重点不是软件命令，而是校准问题中的因果链：天空模型决定预测可见度，仪器增益把模型写成观测数据，求解器通过残差恢复增益，而模型错误会被求解器部分吸收。

习题可分成五个层次。第一层考察阻尼参数与收敛历史；第二层考察参考天线和相位规范；第三层考察模型不完整；第四层考察交替增益更新；第五层要求形成校准诊断结论。所有题目都与 [8.1](8_1_calibration_least_squares_problem.ipynb) 的最小二乘推导对应。


![校准习题使用的统一数据集](figures/calibration_problem_set_dataset.png)

**图 8.P.1** 习题统一数据集的几何和信号结构。左上为五天线东西向阵列，右上为对应的 $uv$ 轨迹，左下为天线 3 的复增益轨迹，右下为一条基线上的模型可见度与受污染数据。


### 统一数据集

习题采用以下设定。天线东西向位置为

$$
(0,36,102,210,348)\ {\rm m},
$$

观测时间覆盖 $-3.5$ 到 $+3.5$ 小时角，天空模型为三个一维点源：

$$
(I_s,l_s)=
(1.0,0),\quad(0.35,0.012),\quad(0.18,-0.021).
$$

理想模型可见度为

$$
m_{pq}(u)=\sum_s I_s\exp(-2\pi iul_s),
$$

观测数据由

$$
d_{pq}(t)=g_p(t)m_{pq}(t)g_q^*(t)+n_{pq}(t)
$$

生成。热噪声幅度约为 0.02 Jy。完整模型和正确方向无关增益求解后，校正残差应接近噪声量级；模型不完整时，残差会停在更高平台。


### A. 阻尼参数与收敛历史

本题考察 Levenberg-Marquardt 风格求解中的阻尼项。设当前参数为 $\boldsymbol{\theta}$，残差局部线性化后，带阻尼的更新可写成

$$
(\mathbf{J}^T\mathbf{J}+\lambda\mathbf{D})\Delta\boldsymbol{\theta}
=-\mathbf{J}^T\boldsymbol{r}.
$$

需要比较至少三组初始阻尼参数 $\lambda_0$，并记录每次迭代后的平均 RMS 残差。分析重点包括：阻尼过小时第一步是否过于激进，阻尼过大时收敛是否明显变慢，以及当前数据集中残差在多少步后进入收益很小的平台。

可采用下列伪代码组织实验：

```python
for lambda0 in [1e-4, 1e-2, 1e0]:
    solved, history = solve_track(
        data_vis, model_vis_true, pairs, nant,
        ref_ant=0, n_iter=12, lambda0=lambda0,
    )
    plot(history.mean(axis=0), label=f"lambda0={lambda0}")
```

结果讨论应明确区分“数值收敛速度”和“最终物理残差”。


### B. 参考天线与规范选择

本题考察全局相位退化。参考天线从 0 改为 2 或 4 后，单天线相位轨迹会发生整体重排；但如果求解正确，改正后的可见度应保持不变。可比较两组校正数据的 RMS 差异：

$$
\Delta_{\rm ref}=\left[\left\langle
|V^{\rm corr}_{\rm ref=0}-V^{\rm corr}_{\rm ref=k}|^2
\right\rangle\right]^{1/2}.
$$

实验结论应说明：参考天线固定的是相位规范，而不是额外的物理信息。参考天线本身若不稳定，会把相位噪声传播到整组解中；但在理想求解下，不同参考选择不应改变校正后的基线可见度。


### C. 模型不完整效应

本题故意从模型中删去第三个弱源，形成

$$
m'_{pq}(u)=1.0+0.35\exp(-2\pi iu\,0.012).
$$

然后用 $m'_{pq}$ 重复增益求解，并比较三类残差：完整模型校准后相对于真模型的残差，不完整模型校准后相对于真模型的残差，以及不完整模型校准后相对于自身模型的残差。

分析重点是求解器的误差吸收。错误模型仍能让目标函数下降，但它不能恢复被删去源的真实可见度贡献。若不完整模型下的增益看起来平滑，而真模型残差仍明显偏高，就说明求解器已经把部分天空结构错误地解释为仪器项。


### D. 交替增益更新

本题实现一种逐天线交替更新的方向无关增益求解器，并与 LM 风格求解比较。固定其他天线当前解时，天线 $p$ 的最小二乘更新可写成

$$
g_p \leftarrow
\frac{\sum_{q\ne p} d_{pq}\,g_q\,m_{pq}^*}
{\sum_{q\ne p}|g_q|^2|m_{pq}|^2}.
$$

每轮更新后需要重新固定参考天线相位。比较内容包括收敛速度、对初始值的敏感性、完整模型下的最终残差，以及不完整模型下是否同样会吸收天空误差。

可采用下列伪代码作为骨架：

```python
def solve_alternating(data, model, pairs, nant, ref_ant=0, n_iter=20):
    gains = np.ones((data.shape[0], nant), dtype=complex)
    for t in range(data.shape[0]):
        gt = np.ones(nant, dtype=complex)
        for _ in range(n_iter):
            for p in range(nant):
                # update g_p using all baselines connected to antenna p
                pass
            gt /= gt[ref_ant] / abs(gt[ref_ant])
        gains[t] = gt
    return gains
```

该题的核心不是得到某个唯一代码实现，而是理解天线基冗余如何让逐天线更新成为可能。


### E. 校准诊断结论

最后形成一段 5 到 8 句话的诊断结论。结论应覆盖以下问题：最小二乘目标函数为什么需要参考天线；为什么“求解器收敛”不等价于“天空模型正确”；模型不完整时，残差、增益轨迹和图像结构会出现哪些迹象；真实数据处理中，哪些证据会提示应改进 sky model，而不是继续调整求解器参数。

一个合格结论应把数值现象和物理解释连接起来。例如，完整模型下残差接近噪声，不完整模型下残差停在更高平台，说明求解器只能最小化给定目标函数，不能替代天空模型判断。若残差围绕特定源或特定方向组织起来，还应进一步考虑方向依赖校准而不是单纯增加 2GC 迭代次数。


### 参考核对量

在与 [8.1](8_1_calibration_least_squares_problem.ipynb) 相同的随机种子和噪声水平下，典型结果应满足以下数量级关系：未校准数据相对于真模型的 RMS 残差约为 $4\times10^{-1}$ Jy，完整模型校准后的 RMS 残差约为 $2\times10^{-2}$ Jy，不完整模型校准后相对于真模型的残差约为 $1\times10^{-1}$ Jy。不同实现的精确数值可以略有差异，但这三个量级关系应保持稳定。
